## Mamba: Linear-Time Sequence Modeling with Selective State Spaces
This notebook presents a from-scratch PyTorch implementation of the Mamba architecture, as introduced in the paper "Mamba: Linear-Time Sequence Modeling with Selective State Spaces" by Albert Gu and Tri Dao. 

references:

- Paper:  [Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752)
- Official Repo: [state-spaces/mamba](https://github.com/state-spaces/mamba)
- Reference MambaViT: [JLrumberger/MambaViT](https://github.com/JLrumberger/MambaViT)
- Reference parallel pure-pytorch implementation: [alxndrTL/mamba.py](https://github.com/alxndrTL/mamba.py)


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as nf
from torch.utils.data import DataLoader

import torchvision.datasets as datasets
import torchvision.transforms as transforms

from einops import rearrange, repeat, einsum
import math

### The State Space Model (SSM)

A State Space Model (SSM) is a system that maps a 1D input sequence $x(t)$ to a 1D output sequence $y(t)$ through an intermediate latent state vector $h(t)$.

1. *Continuous-Time Formulation*

The basic definition of an SSM is a linear Ordinary Differential Equation (ODE):
$$
h'(t) = Ah(t) + Bx(t)
$$
$$
y(t) = Ch(t)
$$

Where:
- $h(t) \in R^N$ is the latent state.
- $x(t) \in R^D$ is the input.
- $y(t) \in R^D$ is the output.
- $A \in R^{N \times N}$ is the **state matrix**.
- $B \in R^{N \times D}$ is the **input matrix**.
- $C \in R^{D \times N}$ is the **output matrix**.

The matrix governs the internal dynamics of the system (how the state evolves on its own), while $B$ and $C$ respectively describe how the system is affected by the input and how it is projected to the output.

2. *Discretization*

This formulation originates from *Control Theory* and describes systems which process continuous input signals. Therefore it must be translated into a discrete form to be computable.

Let us denote $\Delta$ a given *timestep*.

The discrete form of the matrix $A$ and $B$ are obtained using the *Zero-Order Hold (ZOH)* method, which assumes that the input $x(t)$ is held constant over the interval $[t_k, t_{k+1})$ which yields:

$$
\bar{A} = \exp(\Delta A)
$$
$$
\bar{B} = (\Delta A)^{-1}(\exp(\Delta A) - I)B
$$

The discretized SSM can then be written as a standard recurrence:
$$
h_k = \bar{A}h_{k-1} + \bar{B}x_k
$$
$$
y_k = Ch_k
$$

3. *The "Selective" Innovation*

The key innovation of Mamba is making the SSM **selective**. Instead of having fixed $B$, $C$, and $\Delta$ matrices for the entire batch, these parameters are generated *dynamically* from the input sequence $x$.

$$
\Delta = proj_\Delta(x)
$$
$$
B = proj_B(x)
$$
$$
C = proj_C(x)
$$

This means the system can adapt to the content. For each token in the input sequence, Mamba can modulate how much it focuses on or ignores that token by changing $\Delta$ and $B$. This gives the model the ability to filter information.


In [2]:
class MambaInput(nn.Module):
    """
    Project the input from `input_dim` to `2 * `model_dim` and split it into the main data path `x` and the gate `z`.
    """
    def __init__(self, input_dim: int, expand_factor: int= 1):
        super().__init__()
        self.input_dim = input_dim
        self.model_dim = int(expand_factor * input_dim)

        # Linear layer for the main data path
        # self.x_proj = nn.Linear(input_dim, self.model_dim, bias=False)
        
        # Linear layer for the gate path
        # self.z_proj = nn.Linear(input_dim, self.model_dim, bias=False)

        # A single projection layer for both main and gate paths
        self.in_proj = nn.Linear(self.input_dim, 2 * self.model_dim, bias=False)

    def forward(self, x):
        """
        x: (batch_size, seq_len, input_dim)
        """

        x_proj = self.in_proj(x)

        x_main, z_gate = x_proj.chunk(2, dim=-1)
        
        return x_main, z_gate
    
    

### Block 2: 1D Causal Convolution (MambaConv)

This module's purpose is to give the model awareness of the immediate local context of each token before that token is processed by the long-range SSM. 

It is implemented as a 1D Causal Depthwise Convolution.

- **Causal:** We use a standard padding-and-truncation technique (padding=d_conv-1, then slicing the output to the original length L) to ensure that the output at any timestep t only depends on inputs from t and earlier. This prevents any "cheating" by looking at future tokens.

- **Depthwise:** We set groups=d_inner to ensure that each channel is processed independently. This is critical because the subsequent SSM module treats each channel as a separate state machine. This design prevents information from being mixed across channels, preserving the SSM's core operational principle.

This block takes the main data path x from the MambaInput module and outputs a new x of the same shape, but with local information now fused into each token's representation.


In [3]:
class MambaConv(nn.Module):
    """
    Applies a 1D causal depthwise convolution.
    """

    def __init__(self, model_dim: int, conv_dim: int):
        super().__init__()
        self.model_dim = model_dim
        self.conv_dim = conv_dim

        self.conv = nn.Conv1d(
            in_channels=self.model_dim,
            out_channels=self.model_dim,
            bias=True,
            kernel_size=self.conv_dim,
            groups=self.model_dim,
            padding=self.conv_dim - 1,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, L, _ = x.shape

        x_permuted = x.permute(0, 2, 1)

        x_conv = self.conv(x_permuted)[:, :, :L]

        x_conv_permuted = x_conv.permute(0, 2, 1)

        return x_conv_permuted

### Selective Scan

In computer science the scan of a sequence of numbers x0, x1, x2, ... is a second sequence of numbers y0, y1, y2, ... obtained by applying an operator to all previous elements in the input.

For addition, the scan (also prefix sum, cumulative sum or inclusive scan) is:

$$
\begin{align*}
& y0 = x0 \\ 
& y1 = x0 + x1 \\
& y2 = x0 + x1 + x2 \\
& ...
\end{align*}
$$



In prior SSMs (like S4), the state-transition matrices $\bar{A}$ and $\bar{B}$ were static. They were calculated once at the beginning of training and remained fixed for every token in every sequence.

Mamba introduce dynamic and data-dependent. Specifically, the timestep $\delta$ and the input matrix $B$ are generated from the input data.

$$\Delta = proj_{\Delta}(x)$$
$$B = proj_B(x)$$

This enable the SSM to be selective. 
1. **Selective forgetting / Remembering:**
  By changinn the value of $\Delta$, the model modifies the state matrix $\bar{A} = exp(\Delta A)$. A large $\Delta$ can cause the state h to decay quickly, effectively "forgetting" past information. A small $\Delta$ can cause the state to persist, "remembering" information for a long time. The model can learn to make $\Delta$ large for unimportant tokens and small for important ones.
2. **Selective Focusing:**
  By changing the matrix $B$, the model can control how much of the current input $x_k$ is written to the state $h_k$. If the model decides a token is not important it can learn to make the corresponding $B$ values small, effectively ignoring that input. If a token is critical, it can make B large to focus on it.


In [4]:
def selective_scan(x, delta, A, B, C, D):
    """
    Performs the selective scan operation.
    """
    B_batch, L_len, model_dim = x.shape
    N_state = A.shape[1]

    
    # 1. Discretize A and B
    # A_bar = exp(delta * A)
    # B_bar = (exp(delta * A) - 1) / A * B
    
    # Discretize A: (B, L, model_dim, N)
    delta_A = torch.exp(einsum(delta, A, 'b l d, d n -> b l d n'))
    
    # Discretize B: B_bar = delta * B
    # B_bar is an outer product of delta and B, for each item in batch and sequence.
    # (B, L, D, N)
    delta_B = einsum(delta, B, 'b l d, b l n -> b l d n')
    
    # The state update term B_bar * x
    # (B, L, D, N) * (B, L, D, 1) -> (B, L, D, N)
    delta_B_x = delta_B * x.unsqueeze(-1)

    # 2. Perform the scan operation (recurrent loop)
    # Initialize the hidden state h to zeros.
    h = torch.zeros(B_batch, model_dim, N_state, device=x.device)
    
    # A list to store the output y at each timestep.
    ys = []
    
    for i in range(L_len):
        # Get the parameters for the current timestep
        h = delta_A[:, i] * h + delta_B_x[:, i]
        ys.append(h)
    
    # Stack the outputs into a single tensor
    # h_stacked is (L, B, model_dim, N)
    h_stacked = torch.stack(ys, dim=1) # (B, L, model_dim, N)
    
    # 3. Compute the final output y
    # y = C * h
    y = einsum(h_stacked, C, 'b l d n, b l n -> b l d')
    
    # 4. Add the skip connection D * x
    y = y + x * D.unsqueeze(0)
    
    return y


### Block 3: The State Space Model (SSM)

This module takes the output from the MombaConv Block and computes the SSM reccurence. 

It performs this operation in three steps:
1. Projecting the input $x$ to generate the selective parameters $\Delta$ $B$, and $C$.
2. Holding the non-selective parameters A and D.
3. Calling the selective_scan function to perform the main computation.


In [5]:
class SSM(nn.Module):
    def __init__(
        self,
        model_dim: int,
        state_dim: int,
        dt_min=0.001,
        dt_max=0.1,
    ):
        super().__init__()
        self.model_dim = model_dim
        self.state_dim = state_dim

        # Projections for selective parameters, delta, B, and C
        self.delta_proj = nn.Linear(self.model_dim, self.model_dim, bias=True)

        self.B_proj = nn.Linear(self.model_dim, self.state_dim, bias=False)
        self.C_proj = nn.Linear(self.model_dim, self.state_dim, bias=False)

        # Matrix A is not data-dependent
        A = torch.arange(1, self.state_dim + 1, dtype=torch.float32).repeat(
            self.model_dim, 1
        )
        self.A_log = nn.Parameter(torch.log(A))
        self.A_log._no_weight_decay = True

        # Skip connection D.
        self.D = nn.Parameter(torch.ones(self.model_dim))
        self.D._no_weight_decay = True

        nn.init.zeros_(self.delta_proj.weight)
        dt_init = torch.exp(
            torch.rand(self.model_dim) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        inv_dt = dt_init + torch.log(-torch.expm1(-dt_init))
        
        with torch.no_grad():
            self.delta_proj.bias.copy_(inv_dt)

    def forward(self, x: torch.Tensor):
        """
        x: (batch_size, seq_len, model_dim)
        """

        # Project the input
        delta = nf.softplus(self.delta_proj(x))
        B = self.B_proj(x)
        C = self.C_proj(x)

        A = -torch.exp(self.A_log.float())

        D = self.D.float()

        y = selective_scan(x, delta, A, B, C, D)

        return y

### Assembling the full MambaBlock

The MambaBlock assembles the three previous sub-module to define the elemental block of the Mamba architecture and orchestrate the flow of data through each of them.

In [6]:
class MambaBlock(nn.Module):
    def __init__(
        self,
        model_dim: int,
        state_dim: int,
        conv_dim: int,
        expand_factor: int = 1,
    ):
        super().__init__()
        self.model_dim = model_dim
        self.state_dim = state_dim
        self.conv_dim = conv_dim
        self.expand_factor = expand_factor

        self.input_block = MambaInput(model_dim, expand_factor)
        self.conv_block = MambaConv(self.input_block.model_dim, conv_dim)
        self.ssm_block = SSM(self.input_block.model_dim, state_dim)

        self.output_proj = nn.Linear(self.input_block.model_dim, self.model_dim)

        self.norm = nn.LayerNorm(self.model_dim)

    def forward(self, x: torch.Tensor):
        """
        x: (batch_size, seq_len, model_dim)
        """
        residual = x

        x = self.norm(x)

        x_main, z_gate = self.input_block(x)
        x_main = self.conv_block(x_main)
        y_ssm = self.ssm_block(x_main)

        gated_output = y_ssm * nf.silu(z_gate)

        x = self.output_proj(gated_output)

        return x + residual

In [7]:
class MambaVis(nn.Module):
    def __init__(
        self,
        model_dim: int,
        n_layers: int,
        n_classes: int,
        state_dim: int = 16,
        conv_dim: int = 4,
        expand_factor: int = 2,
    ) -> None:
        super().__init__()
        self.model_dim = model_dim
        self.n_layers = n_layers
        self.n_classes = n_classes
        self.state_dim = state_dim
        self.conv_dim = conv_dim
        self.expand_factor = expand_factor

        self.embedding = nn.Linear(1, self.model_dim)

        self.layers = nn.ModuleList(
            [
                MambaBlock(
                    model_dim=model_dim,
                    state_dim=state_dim,
                    conv_dim=conv_dim,
                    expand_factor=expand_factor,
                )
                for _ in range(n_layers)
            ]
        )
        self.norm = nn.LayerNorm(self.model_dim)
        self.head = nn.Linear(self.model_dim, self.n_classes)

    def forward(self, x: torch.Tensor):
        """
        x: (batch_size, seq_len, 1)
        """
        x = x.unsqueeze(-1)
        x = self.embedding(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x[:, -1, :])

        x = self.head(x)

        return x

In [8]:
# --- Hyperparameters ---
dim = 64          # Model dimension
n_layers = 4      # Number of Mamba blocks
n_classes = 10    # Number of output classes (digits 0-9)
batch_size = 64
learning_rate = 1e-3
epochs = 3

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Data Loading ---
transform = transforms.Compose([
    transforms.ToTensor(),
    # Normalize with the standard mean and std for MNIST
    transforms.Normalize((0.1307,), (0.3081,)),
    # Flatten the image from (1, 28, 28) to (784)
    transforms.Lambda(lambda x: x.view(-1))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)



Using device: cuda


In [9]:
# --- Model, Loss, and Optimizer ---
model = MambaVis(
    model_dim=dim,
    n_layers=n_layers,
    n_classes=n_classes
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters.")

Model created with 193674 parameters.


In [10]:
from tqdm import tqdm

# --- Training Loop ---
for epoch in range(epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Forward pass
        outputs = model(inputs)
        
        # 2. Calculate loss
        loss = criterion(outputs, labels)
        
        # 3. Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()

        # Clip gradients to prevent them from exploding
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        
        total_loss += loss.item()
        
        # Update progress bar
        progress_bar.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Average Training Loss: {avg_train_loss:.4f}")

print("Training finished.")

Epoch 1/3: 100%|██████████| 938/938 [12:52:45<00:00, 49.43s/it, loss=0.205]  


Epoch 1/3 - Average Training Loss: 0.7231


Epoch 2/3: 100%|██████████| 938/938 [12:54:44<00:00, 49.56s/it, loss=0.0863]  


Epoch 2/3 - Average Training Loss: 0.1771


Epoch 3/3: 100%|██████████| 938/938 [12:51:54<00:00, 49.38s/it, loss=0.0449]  

Epoch 3/3 - Average Training Loss: 0.1159
Training finished.


In [ ]:
# --- Get a single batch ---
data_iter = iter(train_loader)
inputs, labels = next(data_iter)
inputs, labels = inputs.to(device), labels.to(device)

# --- Training Loop on that single batch ---
model.train()
for i in range(200):  # Train for 200 steps
    # 1. Forward pass
    outputs = model(inputs)
    
    # 2. Calculate loss
    loss = criterion(outputs, labels)
    
    # 3. Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()

    # --- GRADIENT INSPECTION ---
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    
    total_norm = total_norm ** (1. / 2)

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    print(f"Step {i}, Loss: {loss.item()}, Total Norm: {total_norm}")

# If the loss doesn't plummet to near-zero, something is wrong with the model's architecture or gradient flow.

# Parallel scan implementation

To achieve high performance, the original mamba implementation uses a parallel scan algorithm written in CUDA which achieves linear-time scaling on sequence length.

For simplicity and clarity this notebook implement a pure PyTorch parallel scan algorithm based on the mamba.py repository.

Because the SSM rule is associative, the selective scan can be parallelised using Blelloch's algorithm. 

The implementation takes as input two sequences:
- A: the sequence of deltas computed from the sequence of inputs.
- X: the sequence of inputs multiplied by B. 

It then computes the parallel scan of the two sequences with  the operator:

$$
h[t] = A[t] * h[t-1] + X[t]
$$

In [ ]:
class ParallelScan(torch.autograd.Function):
    @staticmethod
    def parallel_scan(A, X):
        """
        A : (B, D, L, N)
        X : (B, D, L, N)
        """

        B, D, L, _ = A.size()
        num_steps = int(math.log2(L))

        # up sweep (last 2 steps unfolded)
        Aa = A 
        Xa = X

        for _ in range(num_steps - 2):
            T = Xa.size(2)
            Aa = Aa.view(B, D, T//2, 2, -1)
            Xa = Xa.view(B, D, T//2, 2, -1)

            Xa[:, :, :, 1].add_(Aa[:, :, :, 1]).mul_(Xa[:, :, :, 0])
            Aa[:, :, :, 1].mul_(Aa[:, :, :, 0])

            Aa = Aa[:, :, :, 1]
            Xa = Xa[:, :, :, 1]

        if Xa.size(2) == 4:
            Xa[:, :, 1].add_(Aa[:, :, 1].mul(Xa[:, :, 0]))
            Aa[:, :, 1].mul_(Aa[:, :, 0])

            Xa[:, :, 3].add_(Aa[:, :, 3].mul(Xa[:, :, 2] + Aa[:, :, 2].mul(Xa[:, :, 1])))
        elif Xa.size(2) == 2:
            Xa[:, :, 1].add_(Aa[:, :, 1].mul(Xa[:, :, 0]))
            return
        else:
            return

        

: 